# Setup
Set up the EOMT Large model (pretrained segmentation head using DinoV3 backbone).

In [ ]:
import os
os.getcwd()

In [ ]:
from pathlib import Path
from PIL import Image
import torch
import numpy as np
import matplotlib.pyplot as plt
from transformers import AutoImageProcessor, AutoModelForUniversalSegmentation

# Config
model_id = "tue-mps/coco_panoptic_eomt_large_640"   # EoMT-L panoptic (640)
# FIXME: ADJUST TO LOCAL IMAGE PATH
# image_path = Path("/home/teun/Projects/maritime-object-detection/2026Q2_innovation_sprint/47b816b2-b314-4489-ba75-06ecfbeb71c8.jpeg")
image_path = Path("../images/cropped_images/bridge_w_persons3_2025_CASE_OP_OS_4_STERE_BOIKY.JPG")
patch_size = 1024  # process image in <=1024x1024 tiles to reduce memory usage
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
out_dir = Path("outputs/eomt_inference")
out_dir.mkdir(parents=True, exist_ok=True)

# Load
processor = AutoImageProcessor.from_pretrained(model_id)
model = AutoModelForUniversalSegmentation.from_pretrained(model_id).to(device)

# Run panoptic segmentation on 1024x1024 patches of the image

In [ ]:
from tqdm.auto import tqdm
# Load image
image = Image.open(image_path).convert("RGB")

# Patch-wise panoptic inference to keep memory bounded
width, height = image.size
stitched_seg_map = np.zeros((height, width), dtype=np.int32)
segments_info_all = []
next_global_id = 1

for top in tqdm(range(0, height, patch_size)):
    for left in tqdm(range(0, width, patch_size), leave=False):
        right = min(left + patch_size, width)
        bottom = min(top + patch_size, height)
        tile = image.crop((left, top, right, bottom))

        inputs = processor(images=tile, return_tensors="pt").to(device)
        with torch.inference_mode():
            outputs = model(**inputs)

        pan = processor.post_process_panoptic_segmentation(
            outputs, target_sizes=[tile.size[::-1]]
        )[0]
        seg_tile = np.array(pan["segmentation"], dtype=np.int32)
        tile_segments = pan.get("segments_info", pan.get("segments", []))

        remapped_tile = np.zeros_like(seg_tile, dtype=np.int32)
        for s in tile_segments:
            sid = s.get("id", s.get("segment_id"))
            if sid is None:
                continue
            gid = next_global_id
            next_global_id += 1
            remapped_tile[seg_tile == sid] = gid

            s_new = dict(s)
            s_new["id"] = gid
            s_new["tile_box_xyxy"] = [left, top, right, bottom]
            segments_info_all.append(s_new)

        stitched_seg_map[top:bottom, left:right] = remapped_tile

# Build color overlay from stitched map
cmap = plt.get_cmap("tab20")
color_img = np.zeros((height, width, 3), dtype=np.float32)
unique_ids = np.unique(stitched_seg_map)
unique_ids = unique_ids[unique_ids != 0]
for i, sid in enumerate(unique_ids):
    color = np.array(cmap(i % cmap.N)[:3])
    color_img[stitched_seg_map == sid] = color

img_arr = np.array(image).astype(np.float32) / 255.0
overlay = np.clip(0.5 * color_img + 0.5 * img_arr, 0, 1)

# Show
plt.figure(figsize=(14, 6))
plt.subplot(1, 2, 1); plt.imshow(image); plt.axis("off"); plt.title("Original")
plt.subplot(1, 2, 2); plt.imshow(overlay); plt.axis("off"); plt.title("Panoptic overlay (patch-wise)")

# Save results
from PIL import Image as PILImage
PILImage.fromarray((overlay * 255).astype(np.uint8)).save(
    out_dir / "eomt_large_panoptic_overlay_patches.png"
)
np.save(out_dir / "eomt_large_panoptic_map_patches.npy", stitched_seg_map)
print("Saved patch-wise overlay and segmap to", out_dir)
print("Total segments:", len(segments_info_all))

# First segment the boat, then try to segment objects on the boat

In [ ]:
from tqdm.auto import tqdm
# Two-pass boat-focused patch inference
first_pass_max_side = 1024  # downsample size for coarse boat localization
boat_patch_size = 1024      # patch size for fine pass
boat_patch_stride = 1024    # set < boat_patch_size for overlap

# 1) First pass on a downsampled image to localize the boat
full_image = Image.open(image_path).convert("RGB")
full_w, full_h = full_image.size
scale = min(1.0, first_pass_max_side / max(full_w, full_h))
small_size = (max(1, int(full_w * scale)), max(1, int(full_h * scale)))
small_image = full_image.resize(small_size, Image.Resampling.LANCZOS)
print(f"First pass image size: {small_size} (from {(full_w, full_h)})")

inputs_small = processor(images=small_image, return_tensors="pt").to(device)
with torch.inference_mode():
    outputs_small = model(**inputs_small)

pan_small = processor.post_process_panoptic_segmentation(
    outputs_small, target_sizes=[small_image.size[::-1]]
)[0]
seg_small = np.array(pan_small["segmentation"], dtype=np.int32)
segments_small = pan_small.get("segments_info", pan_small.get("segments", []))

id2label = getattr(model.config, "id2label", {})
def label_name_from_id(label_id):
    label_id_int = int(label_id)
    return str(id2label.get(label_id_int, id2label.get(str(label_id_int), label_id_int)))

boat_segment_ids_small = []
for seg in segments_small:
    label_id = seg.get("label_id", seg.get("category_id", seg.get("class_id", -1)))
    label_name = label_name_from_id(label_id).lower()
    if "boat" in label_name or "ship" in label_name:
        sid = seg.get("id", seg.get("segment_id"))
        if sid is not None:
            boat_segment_ids_small.append(int(sid))

if len(boat_segment_ids_small) == 0:
    raise RuntimeError(
        "No boat/ship segment found in first pass. Try another image, a larger first_pass_max_side, or inspect model labels."
    )

boat_mask_small = np.isin(seg_small, np.array(boat_segment_ids_small, dtype=np.int32))
boat_mask_full = np.array(
    Image.fromarray((boat_mask_small * 255).astype(np.uint8)).resize((full_w, full_h), Image.Resampling.NEAREST)
).astype(bool)

ys, xs = np.where(boat_mask_full)
if len(xs) == 0 or len(ys) == 0:
    raise RuntimeError("Boat mask is empty after upscaling. Cannot continue with patch pass.")

x_min, x_max = xs.min(), xs.max()
y_min, y_max = ys.min(), ys.max()
print(f"Boat ROI in full image: x=[{x_min},{x_max}], y=[{y_min},{y_max}]")

# 2) Fine pass on 1024x1024 patches over only the boat ROI, background blacked out
full_arr = np.array(full_image)
refined_seg = np.zeros((full_h, full_w), dtype=np.int32)
segments_refined = []
next_global_id = 1

for top in tqdm(range(int(y_min), int(y_max) + 1, boat_patch_stride)):
    for left in tqdm(range(int(x_min), int(x_max) + 1, boat_patch_stride), leave=False):
        bottom = min(top + boat_patch_size, full_h)
        right = min(left + boat_patch_size, full_w)

        boat_mask_patch = boat_mask_full[top:bottom, left:right]
        if not boat_mask_patch.any():
            continue

        tile_arr = full_arr[top:bottom, left:right].copy()
        tile_arr[~boat_mask_patch] = 0  # blackout non-boat background
        tile_img = Image.fromarray(tile_arr)

        inputs_tile = processor(images=tile_img, return_tensors="pt").to(device)
        with torch.inference_mode():
            outputs_tile = model(**inputs_tile)

        pan_tile = processor.post_process_panoptic_segmentation(
            outputs_tile, target_sizes=[tile_img.size[::-1]]
        )[0]
        seg_tile = np.array(pan_tile["segmentation"], dtype=np.int32)
        tile_segments = pan_tile.get("segments_info", pan_tile.get("segments", []))

        remapped_tile = np.zeros_like(seg_tile, dtype=np.int32)
        for seg in tile_segments:
            sid = seg.get("id", seg.get("segment_id"))
            if sid is None:
                continue

            pixel_mask = (seg_tile == sid) & boat_mask_patch
            if not pixel_mask.any():
                continue

            gid = next_global_id
            next_global_id += 1
            remapped_tile[pixel_mask] = gid

            seg_new = dict(seg)
            seg_new["id"] = gid
            seg_new["tile_box_xyxy"] = [int(left), int(top), int(right), int(bottom)]
            segments_refined.append(seg_new)

        write_region = refined_seg[top:bottom, left:right]
        write_region[(write_region == 0) & (remapped_tile > 0)] = remapped_tile[(write_region == 0) & (remapped_tile > 0)]
        refined_seg[top:bottom, left:right] = write_region

# 3) Visualize refined output
cmap = plt.get_cmap("tab20")
color_img = np.zeros((full_h, full_w, 3), dtype=np.float32)
unique_ids = np.unique(refined_seg)
unique_ids = unique_ids[unique_ids != 0]
for i, sid in enumerate(unique_ids):
    color_img[refined_seg == sid] = np.array(cmap(i % cmap.N)[:3])

img_arr = full_arr.astype(np.float32) / 255.0
overlay = np.clip(0.55 * color_img + 0.45 * img_arr, 0, 1)

plt.figure(figsize=(16, 6))
plt.subplot(1, 3, 1); plt.imshow(full_image); plt.axis("off"); plt.title("Original")
plt.subplot(1, 3, 2); plt.imshow(boat_mask_full, cmap="gray"); plt.axis("off"); plt.title("Boat mask (1st pass)")
plt.subplot(1, 3, 3); plt.imshow(overlay); plt.axis("off"); plt.title("Boat-focused patch inference")
plt.tight_layout()
plt.show()

from PIL import Image as PILImage
PILImage.fromarray((overlay * 255).astype(np.uint8)).save(
    out_dir / "eomt_boat_focused_patch_overlay.png"
 )
np.save(out_dir / "eomt_boat_focused_patch_map.npy", refined_seg)
print("Saved boat-focused outputs to", out_dir)
print("Refined segments:", len(segments_refined))

# Set up interactive image
Clicking on the image returns the clicked object detected using segmentation

In [ ]:
import matplotlib
matplotlib.use("module://ipympl.backend_nbagg", force=True)

In [ ]:
# Load saved segmentation outputs to avoid rerunning inference after kernel restart
from pathlib import Path
import numpy as np
from PIL import Image

# Prefer existing paths from earlier cells when available
if "out_dir" not in globals():
    out_dir = Path("outputs/eomt_inference")
if "image_path" not in globals():
    image_path = Path(
        "/home/teun/Projects/maritime-object-detection/2026Q2_innovation_sprint/47b816b2-b314-4489-ba75-06ecfbeb71c8.jpeg"
    )

# Candidate maps in priority order (boat-focused first)
map_candidates = [
    out_dir / "eomt_boat_focused_patch_map.npy",
    out_dir / "eomt_large_panoptic_map_patches.npy",
    out_dir / "eomt_large_panoptic_map.npy",
]

map_path = next((p for p in map_candidates if p.exists()), None)
if map_path is None:
    raise FileNotFoundError(
        f"No saved segmentation map found in {out_dir}. Run a segmentation cell first."
    )

seg_loaded = np.load(map_path)

# Expose variables expected by the click-loop cell
if "boat_focused" in map_path.name:
    refined_seg = seg_loaded
    print(f"Loaded boat-focused map into 'refined_seg': {map_path}")
else:
    stitched_seg_map = seg_loaded
    print(f"Loaded patch map into 'stitched_seg_map': {map_path}")

# Load base image used by the click loop
if not Path(image_path).exists():
    raise FileNotFoundError(f"Image not found: {image_path}")

loaded_image = Image.open(image_path).convert("RGB")
full_image = loaded_image
image = loaded_image

# Optional sanity check
h, w = seg_loaded.shape[:2]
img_w, img_h = loaded_image.size
print(f"Seg map shape: {(h, w)}")
print(f"Image size: {(img_h, img_w)}")
if (h, w) != (img_h, img_w):
    print("Warning: map and image dimensions differ; click-loop cell may raise shape mismatch.")
else:
    print("Seg map and image dimensions match. You can run the click-loop cell now.")

In [ ]:
# Persistent interactive click viewer: click any pixel to highlight its segment
import matplotlib.pyplot as plt
import numpy as np

# Prefer the boat-focused map; fall back to patch-wise map if needed
if "refined_seg" in globals() and refined_seg is not None:
    seg_for_click = refined_seg
elif "stitched_seg_map" in globals() and stitched_seg_map is not None:
    seg_for_click = stitched_seg_map
else:
    raise RuntimeError(
        "No segmentation map found. Run Cell 14 (or Cell 13) first to generate a segmentation map."
    )

# Prefer full_image from the two-pass workflow; otherwise use image
if "full_image" in globals() and full_image is not None:
    base_img = np.array(full_image).astype(np.float32) / 255.0
elif "image" in globals() and image is not None:
    base_img = np.array(image).astype(np.float32) / 255.0
else:
    raise RuntimeError("No base image found. Run Cell 14 (or Cell 13) first.")

if seg_for_click.shape[:2] != base_img.shape[:2]:
    raise RuntimeError(
        f"Shape mismatch: seg map has {seg_for_click.shape[:2]}, image has {base_img.shape[:2]}."
    )

# Build one persistent figure and update it on each click
fig, axs = plt.subplots(1, 2, figsize=(14, 6))
axs[0].imshow(base_img)
axs[0].set_title("Click on object")
axs[0].axis("off")

# Right panel starts as original image; updates after each click
overlay_img = axs[1].imshow(base_img.copy())
axs[1].set_title("Highlighted segment")
axs[1].axis("off")

# Click marker on the left image
click_marker = axs[0].scatter([], [], c="yellow", s=80, edgecolors="black")

print("Interactive viewer started.")
print("Click on the left image to highlight a segment. Close the figure to stop.")

def _on_click(event):
    if event.inaxes is not axs[0] or event.xdata is None or event.ydata is None:
        return

    x_i = int(np.clip(round(event.xdata), 0, seg_for_click.shape[1] - 1))
    y_i = int(np.clip(round(event.ydata), 0, seg_for_click.shape[0] - 1))
    selected_id = int(seg_for_click[y_i, x_i])

    # Move/update click marker
    click_marker.set_offsets(np.array([[x_i, y_i]], dtype=np.float32))

    if selected_id == 0:
        axs[1].set_title(f"Background at ({x_i}, {y_i}) - try another point")
        overlay_img.set_data(base_img)
        fig.canvas.draw_idle()
        return

    mask = seg_for_click == selected_id
    area = int(mask.sum())

    # Green overlay for selected segment
    overlay = base_img.copy()
    overlay[mask] = 0.35 * overlay[mask] + 0.65 * np.array([0.0, 1.0, 0.0], dtype=np.float32)

    # Draw thin red boundary
    boundary = np.zeros(mask.shape, dtype=bool)
    boundary[1:, :] |= mask[1:, :] != mask[:-1, :]
    boundary[:-1, :] |= mask[1:, :] != mask[:-1, :]
    boundary[:, 1:] |= mask[:, 1:] != mask[:, :-1]
    boundary[:, :-1] |= mask[:, 1:] != mask[:, :-1]
    overlay[boundary] = np.array([1.0, 0.0, 0.0], dtype=np.float32)

    overlay_img.set_data(overlay)
    axs[1].set_title(f"Segment ID={selected_id}, area={area} px")
    fig.canvas.draw_idle()

cid = fig.canvas.mpl_connect("button_press_event", _on_click)
plt.tight_layout()
plt.show()

# Keep references around so callbacks are not garbage-collected in some backends
_interactive_click_figure = fig
_interactive_click_callback_id = cid

# Attempt the segmentation using SAM (Segment Anything Model)

In [ ]:
# SAM click-to-segment pipeline (independent from EOMT segmentation map)
from pathlib import Path
import numpy as np
import torch
import matplotlib.pyplot as plt
from PIL import Image
from transformers import SamModel, SamProcessor

# Config
sam_model_id = "facebook/sam-vit-base"  # base is lighter; try large/huge for higher quality
sam_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Use current notebook image path if available
if "image_path" not in globals():
    image_path = Path(
        "/home/teun/Projects/maritime-object-detection/2026Q2_innovation_sprint/47b816b2-b314-4489-ba75-06ecfbeb71c8.jpeg"
    )

sam_image = Image.open(image_path).convert("RGB")
sam_base = np.array(sam_image).astype(np.float32) / 255.0

print(f"Loading SAM model: {sam_model_id}")
sam_processor = SamProcessor.from_pretrained(sam_model_id)
sam_model = SamModel.from_pretrained(sam_model_id).to(sam_device).eval()

In [ ]:
# SAM interactive viewer (force interactive backend in VS Code)
from IPython import get_ipython
import matplotlib
import numpy as np
import torch

ip = get_ipython()
if ip is not None:
    try:
        ip.run_line_magic("matplotlib", "ipympl")
    except Exception:
        try:
            ip.run_line_magic("matplotlib", "widget")
        except Exception:
            pass

# Precompute image embeddings once for fast click updates
img_inputs = sam_processor(images=sam_image, return_tensors="pt").to(sam_device)
with torch.inference_mode():
    image_embeddings = sam_model.get_image_embeddings(img_inputs["pixel_values"])

orig_sizes = img_inputs["original_sizes"].cpu()
reshaped_sizes = img_inputs["reshaped_input_sizes"].cpu()

fig, axs = plt.subplots(1, 2, figsize=(14, 6))
axs[0].imshow(sam_base)
axs[0].set_title("SAM: click positive point")
axs[0].axis("off")

overlay_artist = axs[1].imshow(sam_base.copy())
axs[1].set_title("SAM mask")
axs[1].axis("off")
click_marker = axs[0].scatter([], [], c="yellow", s=90, edgecolors="black")

print("SAM interactive viewer started.")
print("Click left image to generate a fresh SAM mask. Close figure to stop.")

def _sam_on_click(event):
    if event.inaxes is not axs[0] or event.xdata is None or event.ydata is None:
        return

    x_i = int(np.clip(round(event.xdata), 0, sam_base.shape[1] - 1))
    y_i = int(np.clip(round(event.ydata), 0, sam_base.shape[0] - 1))
    click_marker.set_offsets(np.array([[x_i, y_i]], dtype=np.float32))

    # Build SAM prompt tensors directly to avoid processor image-preprocess path
    input_points = torch.tensor([[[[x_i, y_i]]]], dtype=torch.float32, device=sam_device)
    input_labels = torch.tensor([[[1]]], dtype=torch.int64, device=sam_device)

    with torch.inference_mode():
        outputs = sam_model(
            image_embeddings=image_embeddings,
            input_points=input_points,
            input_labels=input_labels,
            multimask_output=True,
        )

    masks = sam_processor.image_processor.post_process_masks(
        outputs.pred_masks.cpu(),
        orig_sizes,
        reshaped_sizes,
    )

    # Convert to (num_masks, H, W) robustly
    mask_arr = np.array(masks[0])
    while mask_arr.ndim > 3:
        mask_arr = mask_arr[0]

    scores = outputs.iou_scores.detach().cpu().numpy().reshape(-1)
    n_masks = mask_arr.shape[0]
    best_idx = int(np.argmax(scores[:n_masks]))
    best_mask = mask_arr[best_idx] > 0

    overlay = sam_base.copy()
    overlay[best_mask] = 0.35 * overlay[best_mask] + 0.65 * np.array([0.0, 1.0, 0.0], dtype=np.float32)

    boundary = np.zeros(best_mask.shape, dtype=bool)
    boundary[1:, :] |= best_mask[1:, :] != best_mask[:-1, :]
    boundary[:-1, :] |= best_mask[1:, :] != best_mask[:-1, :]
    boundary[:, 1:] |= best_mask[:, 1:] != best_mask[:, :-1]
    boundary[:, :-1] |= best_mask[:, 1:] != best_mask[:, :-1]
    overlay[boundary] = np.array([1.0, 0.0, 0.0], dtype=np.float32)

    overlay_artist.set_data(overlay)
    axs[1].set_title(f"SAM mask (click=({x_i}, {y_i}), score={scores[best_idx]:.3f})")
    fig.canvas.draw_idle()

sam_cid = fig.canvas.mpl_connect("button_press_event", _sam_on_click)
plt.tight_layout()
plt.show()

# Keep references alive in notebook scope
_sam_interactive_figure = fig
_sam_interactive_callback_id = sam_cid
_sam_model = sam_model
_sam_processor = sam_processor
_sam_image_embeddings = image_embeddings

# Segment a single patch

In [ ]:
# Segment a single already-cropped image (no tiling)

# Config: set this to your cropped image
single_image_path = Path("images/bridge_2022_06B_STERE_SOOBRAZITELNIY_HJQ_2938.JPG")

def segment_image(single_image_path):
    if not single_image_path.exists():
        raise FileNotFoundError(f"Image not found: {single_image_path}")

    # Inference
    single_image = Image.open(single_image_path).convert("RGB")
    inputs = processor(images=single_image, return_tensors="pt").to(device)
    with torch.inference_mode():
        outputs = model(**inputs)

    pan = processor.post_process_panoptic_segmentation(
        outputs, target_sizes=[single_image.size[::-1]]
    )[0]
    seg_map = np.array(pan["segmentation"], dtype=np.int32)
    segments_info = pan.get("segments_info", pan.get("segments", []))

    # Build color overlay
    h, w = seg_map.shape
    cmap = plt.get_cmap("tab20")
    color_img = np.zeros((h, w, 3), dtype=np.float32)
    unique_ids = np.unique(seg_map)
    unique_ids = unique_ids[unique_ids != 0]

    for i, sid in enumerate(unique_ids):
        color_img[seg_map == sid] = np.array(cmap(i % cmap.N)[:3])

    img_arr = np.array(single_image).astype(np.float32) / 255.0
    overlay = np.clip(0.55 * color_img + 0.45 * img_arr, 0, 1)

    # Show
    plt.figure(figsize=(14, 6))
    plt.subplot(1, 2, 1); plt.imshow(single_image); plt.axis("off"); plt.title("Single cropped image")
    plt.subplot(1, 2, 2); plt.imshow(overlay); plt.axis("off"); plt.title("Panoptic overlay (single image)")
    plt.tight_layout()
    plt.show()

    # Save
    from PIL import Image as PILImage
    PILImage.fromarray((overlay * 255).astype(np.uint8)).save(
        out_dir / "eomt_single_image_overlay.png"
    )
    np.save(out_dir / "eomt_single_image_map.npy", seg_map)
    print("Saved single-image overlay and segmap to", out_dir)
    print("Total segments:", len(segments_info))

In [ ]:
import os
from tqdm.auto import tqdm
for image in tqdm(os.listdir("images")):
        segment_image(Path("images") / image)